# 06 — Fulfillment & Delivery Analysis

Olist E-Commerce Python Analytics

Covers the logistics pipeline itself: where delivery time gets spent, how reliable it is, whether geography changes it, and whether performance/estimates shifted over time. Full write-up of every finding will live in `06_fulfillment_and_delivery_analysis_log.md` — comments here are kept short.

## Setup

In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv('../data/cleaned/olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
orders = pd.read_csv('../data/cleaned/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/cleaned/olist_order_items_dataset.csv')
sellers = pd.read_csv('../data/cleaned/olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})

# Parse date columns so comparisons below are apples-to-apples
order_date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                    'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders['purchaseMonth'] = orders['order_purchase_timestamp'].dt.to_period('M')


## Section A — Delivery Timeline Breakdown

#### Q1-Q3. The three delivery legs — first look

In [2]:
a = orders['order_approved_at'] - orders['order_purchase_timestamp']
a.sort_values(ascending=True).dropna()

a = orders['order_delivered_carrier_date'] - orders['order_approved_at']
a.sort_values(ascending=True).dropna()

a = orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']
a.sort_values(ascending=True).dropna()


18600     0 days 00:00:00
84425     0 days 00:00:00
86460     0 days 00:00:00
4825      0 days 00:00:00
3706      0 days 00:00:00
               ...       
61610   188 days 02:12:42
70307   190 days 05:39:15
89130   194 days 05:06:44
19590   195 days 00:48:53
55619   205 days 04:35:00
Length: 96276, dtype: timedelta64[us]

#### Data-quality check — carrier-before-approval and missing dates

In [3]:
badCarrierOrder = orders[orders['order_delivered_carrier_date'] < orders['order_approved_at']]
print(badCarrierOrder.shape[0])
print((orders['order_approved_at'] - orders['order_delivered_carrier_date']).describe())
# Finding: 1,183 rows (~1.2%) have carrier pickup logged before approval.


1183
count                       97445
mean     -3 days +04:23:56.804075
std        3 days 11:52:27.611991
min           -126 days +05:41:54
25%             -4 days +09:59:57
50%             -2 days +04:15:35
75%             -1 days +02:53:47
max               4 days 02:19:27
dtype: object


In [4]:
print(orders['order_approved_at'].isna().sum())
print(orders['order_delivered_carrier_date'].isna().sum())
print(orders['order_delivered_customer_date'].isna().sum())
# Finding: 336 / 1,982 / 2,988 missing - increasing down the pipeline, as expected for orders that didn't progress that far.


336
1982
2988


In [5]:
print(badCarrierOrder['order_status'].value_counts())
# Finding: 1,174 of the 1,183 (99.2%) are 'delivered' orders - this is timestamp-ordering noise on successful orders, not a sign of broken ones.


order_status
delivered    1174
shipped         9
Name: count, dtype: int64


In [6]:
print(orders[orders['order_approved_at'].isna()]['order_status'].value_counts())
print(orders[orders['order_delivered_carrier_date'].isna()]['order_status'].value_counts())
print(orders[orders['order_delivered_customer_date'].isna()]['order_status'].value_counts())
# Finding: NaNs mostly line up with non-delivered statuses as expected. One
# genuine small gap in each: 190 / 201 / 31 'delivered' orders are missing a
# timestamp despite completing - under 0.3% each, worth a footnote only.


order_status
delivered    190
canceled     141
created        5
Name: count, dtype: int64
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
delivered      201
created          5
approved         2
Name: count, dtype: int64
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered        31
created           5
approved          2
Name: count, dtype: int64


#### Q1-Q3 (continued). Leg durations — mean vs. median

In [7]:
approvalLeg = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dropna()
carrierLeg = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dropna()
deliveryLeg = (orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dropna()

print(approvalLeg.describe())
print(carrierLeg.describe())
print(deliveryLeg.describe())
# Finding: all three legs are right-skewed. Leg 1 (approval) worst by far -
# median 20min vs mean 10h12min (~30x). Leg 2 median 1d19h vs mean 2d19h.
# Leg 3 median 7d2h vs mean 9d8h. Median is the honest "typical" figure for
# every leg, especially Leg 1.


count                     99105
mean     0 days 10:12:31.714282
std      1 days 01:26:49.154930
min             0 days 00:00:00
25%             0 days 00:12:54
50%             0 days 00:20:33
75%             0 days 14:24:14
max           187 days 21:10:50
dtype: object
count                     97445
mean     2 days 19:36:03.195925
std      3 days 11:52:27.611991
min           -5 days +21:40:33
25%             0 days 21:06:13
50%             1 days 19:44:25
75%             3 days 14:00:03
max           125 days 18:18:06
dtype: object
count                     96276
mean     9 days 08:01:58.507478
std      8 days 17:53:43.949548
min             0 days 00:00:00
25%             4 days 02:29:57
50%      7 days 02:24:55.500000
75%            12 days 00:47:45
max           205 days 04:35:00
dtype: object


#### Q4. Which leg dominates total time, and which is most variable?

In [8]:
# Finding: Leg 3 (carrier -> customer) dominates total delivery time
# (median ~7 days vs ~20min for Leg 1 and ~1.8 days for Leg 2) and has the
# largest ABSOLUTE variability (std ~8.8 days). Leg 1 has the highest
# RELATIVE variability - its std (~1 day) is ~75x its own median (20min),
# confirming it's the most outlier-driven leg despite being shortest on
# average.


## Section B — On-Time vs. Late Delivery, Overall

#### Q5. On-time vs. late delivery rate

In [9]:
deliveredOrders = orders[orders['order_status'] == 'delivered']
lateRate = deliveredOrders['lateDelivery'].value_counts(normalize=True) * 100
print(lateRate)
# Finding: 91.86% on-time, 8.11% late, 0.03% unknown.


lateDelivery
On Time Delivery    91.856174
Late Delivery        8.111694
Unknown              0.032132
Name: proportion, dtype: float64


#### Q6. Among late orders, how severe is the delay?

In [10]:
lateOrders = deliveredOrders[deliveredOrders['lateDelivery'] == 'Late Delivery']
print(lateOrders['delivery_delay_days'].describe())
# Finding: median 5 days, mean 8.87 days (~77% higher) - right-skewed, same as everywhere else in this project.


count    7826.000000
mean        8.866854
std        13.950287
min         0.000000
25%         1.000000
50%         5.000000
75%        11.000000
max       188.000000
Name: delivery_delay_days, dtype: float64


In [11]:
lateOrders.sort_values('delivery_delay_days', ascending=False).head(10)
# Finding: the most extreme late orders (150-188 days) share near-identical
# order_delivered_customer_date values (several land on 2017-09-19) - a
# batch data-entry artifact, not genuine individual lateness. Flag as a
# data-quality note rather than real extreme delay.


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,lateDelivery,validDateCheck,delivery_days,delivery_delay_days,purchaseMonth
55619,1b3190b2dfa9d789e1f14c05b647a14a,d306426abe5fca15e54b645e4462dc7b,delivered,2018-02-23 14:57:35,2018-02-23 15:16:14,2018-02-26 18:49:07,2018-09-19 23:24:07,2018-03-15,Late Delivery,Valid,208.0,188.0,2018-02
19590,ca07593549f1816d26a572e06dc1eab6,75683a92331068e2d281b11a7866ba44,delivered,2017-02-21 23:31:27,2017-02-23 02:35:15,2017-03-08 13:47:46,2017-09-19 14:36:39,2017-03-22,Late Delivery,Valid,209.0,181.0,2017-02
11399,47b40429ed8cce3aee9199792275433f,cb2caaaead400c97350c37a3fc536867,delivered,2018-01-03 09:44:01,2018-01-03 10:31:15,2018-02-06 01:48:28,2018-07-13 20:51:31,2018-01-19,Late Delivery,Valid,191.0,175.0,2018-01
81401,2fe324febf907e3ea3f2aa9650869fa5,65b14237885b3972ebec28c0f7dd2220,delivered,2017-03-13 20:17:10,2017-03-13 20:17:10,2017-03-17 07:23:53,2017-09-19 17:00:07,2017-04-05,Late Delivery,Valid,189.0,167.0,2017-03
89130,285ab9426d6982034523a855f55a885e,9cf2c3fa2632cee748e1a59ca9d09b21,delivered,2017-03-08 22:47:40,2017-03-08 22:47:40,2017-03-09 08:53:20,2017-09-19 14:00:04,2017-04-06,Late Delivery,Valid,194.0,166.0,2017-03
61610,440d0d17af552815d15a9e41abe49359,7815125148cfa1e8c7fee1ff7974f16c,delivered,2017-03-07 23:59:51,2017-03-09 01:11:33,2017-03-15 13:00:08,2017-09-19 15:12:50,2017-04-07,Late Delivery,Valid,195.0,165.0,2017-03
68769,c27815f7e3dd0b926b58552628481575,f85e9ec0719b16dc4dd0edd438793553,delivered,2017-03-15 23:23:17,2017-03-15 23:23:17,2017-03-21 14:50:25,2017-09-19 17:14:25,2017-04-10,Late Delivery,Valid,187.0,162.0,2017-03
40847,d24e8541128cea179a11a65176e0a96f,beeda72b31be3b8a38b5c2b77d7705c4,delivered,2017-06-12 13:14:11,2017-06-12 13:25:11,2017-06-16 14:44:37,2017-12-04 18:36:29,2017-06-26,Late Delivery,Valid,175.0,161.0,2017-06
38509,0f4519c5f1c541ddec9f21b3bddd533a,1a8a4a30dc296976717f44e7801fdeef,delivered,2017-03-09 13:26:57,2017-03-09 13:26:57,2017-03-22 05:28:14,2017-09-19 14:38:21,2017-04-11,Late Delivery,Valid,194.0,161.0,2017-03
54480,2d7561026d542c8dbd8f0daeadf67a43,8199345f57c6d1cbe9701f92481beb8d,delivered,2017-03-15 11:24:27,2017-03-15 11:24:27,2017-03-16 09:29:16,2017-09-19 14:38:18,2017-04-13,Late Delivery,Valid,188.0,159.0,2017-03


#### Q7. Has the on-time rate changed over time?

In [12]:
monthlyLateRate = deliveredOrders.groupby('purchaseMonth')['lateDelivery'].apply(lambda x: (x == 'Late Delivery').mean() * 100)
print(monthlyLateRate)

print(deliveredOrders.groupby('purchaseMonth')['order_id'].count())
# Finding: 2016-09/10/12 show extreme rates but are near-empty ramp-up
# months (as few as 1 delivered order) - noise. Among months with real
# volume, late-rate sits in a 3-8% band through most of 2017, but spikes to
# 14.3% (Nov 2017) and 16.0%/21.4% (Feb/Mar 2018) - all high-volume months
# (6500-7300 delivered orders), so these are genuine signals, not
# small-sample noise. Nov 2017 lines up with Black Friday-season demand;
# Feb/Mar 2018 has no obvious seasonal driver.


purchaseMonth
2016-09    100.000000
2016-10      1.132075
2016-12      0.000000
2017-01      3.066667
2017-02      3.206292
2017-03      5.577376
2017-04      7.859314
2017-05      3.609701
2017-06      3.859649
2017-07      3.434917
2017-08      3.315049
2017-09      5.204819
2017-10      5.292541
2017-11     14.309233
2017-12      8.380192
2018-01      6.563870
2018-02     15.987796
2018-03     21.362273
2018-04      5.310385
2018-05      8.238258
2018-06      1.360879
2018-07      4.481247
2018-08     10.392064
Freq: M, Name: lateDelivery, dtype: float64
purchaseMonth
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M, Name: order_id, dtype: int64


#### Population check before Q8 — order_status by month

In [13]:
statusByMonth = orders.groupby(['purchaseMonth', 'order_status']).size().unstack()
print(statusByMonth)
# Finding: 2018-09/10 have ZERO delivered orders - must be excluded outright
# from any delivered-only trend, not just footnoted. 2016-09/12 also have
# only 1 delivered order each.


order_status   approved  canceled  created  delivered  invoiced  processing  \
purchaseMonth                                                                 
2016-09             NaN       2.0      NaN        1.0       NaN         NaN   
2016-10             NaN      24.0      NaN      265.0      18.0         2.0   
2016-12             NaN       NaN      NaN        1.0       NaN         NaN   
2017-01             NaN       3.0      NaN      750.0      12.0         9.0   
2017-02             1.0      17.0      NaN     1653.0      11.0        32.0   
2017-03             NaN      33.0      NaN     2546.0       3.0        23.0   
2017-04             1.0      18.0      NaN     2303.0      14.0        10.0   
2017-05             NaN      29.0      NaN     3546.0      16.0        23.0   
2017-06             NaN      16.0      NaN     3135.0      11.0        12.0   
2017-07             NaN      28.0      NaN     3872.0       7.0        11.0   
2017-08             NaN      27.0      NaN     4193.

#### Q8. Is Olist's delivery estimate getting more or less accurate over time?

In [14]:
deliveredOrders = orders[orders['order_status'] == 'delivered'].copy()
deliveredOrders['estimateAccuracyDays'] = (deliveredOrders['order_delivered_customer_date'] - deliveredOrders['order_estimated_delivery_date']).dt.days

monthlyEstimateGap = deliveredOrders.groupby('purchaseMonth')['estimateAccuracyDays'].mean()
print(monthlyEstimateGap)
# Finding: negative = delivered early. Excluding 2016-09/12 and 2018-09/10
# (confirmed near-zero/zero delivered volume above), the early-delivery
# buffer shrinks from ~-19/-27 days in early 2017 to ~-6/-13 days by 2018.


purchaseMonth
2016-09    36.000000
2016-10   -36.632184
2016-12   -22.000000
2017-01   -27.363151
2017-02   -19.189582
2017-03   -12.326130
2017-04   -12.971776
2017-05   -13.495767
2017-06   -12.642629
2017-07   -12.454286
2017-08   -13.094011
2017-09   -11.348193
2017-10   -11.923403
2017-11    -8.123491
2017-12   -13.008525
2018-01   -12.934361
2018-02    -8.290770
2018-03    -6.427674
2018-04   -12.895116
2018-05   -12.190991
2018-06   -19.244094
2018-07   -11.441520
2018-08    -8.162179
Freq: M, Name: estimateAccuracyDays, dtype: float64


In [15]:
monthlyDeliveryDays = deliveredOrders.groupby('purchaseMonth')['delivery_days'].mean()
print(monthlyDeliveryDays)
# Finding: actual delivery_days did NOT increase over the same window - it
# dropped to 7-9 days by mid/late 2018 (from 11-15 days in 2017). So the
# shrinking estimate-buffer reflects Olist tightening its ESTIMATES while
# real delivery performance was simultaneously improving - not deliveries
# getting slower and eating into a fixed buffer.


purchaseMonth
2016-09    54.000000
2016-10    19.214559
2016-12     4.000000
2017-01    12.084112
2017-02    12.615990
2017-03    12.398428
2017-04    14.352584
2017-05    10.762698
2017-06    11.506382
2017-07    11.152292
2017-08    10.704605
2017-09    11.400723
2017-10    11.416927
2017-11    14.699506
2017-12    14.936695
2018-01    13.637290
2018-02    16.508619
2018-03    15.867485
2018-04    11.048544
2018-05    10.959105
2018-06     8.774442
2018-07     8.503736
2018-08     7.286412
Freq: M, Name: delivery_days, dtype: float64


## Section C — Geographic Fulfillment Patterns

#### Q9. Does a same-state order deliver faster than a cross-state one?

In [16]:
orderItemsSeller = order_items.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')
orderGeo = orderItemsSeller.merge(
    orders[['order_id', 'customer_id', 'delivery_days', 'delivery_delay_days', 'order_status']],
    on='order_id', how='left'
)
orderGeo = orderGeo.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')

deliveredGeo = orderGeo[orderGeo['order_status'] == 'delivered'].copy()
deliveredGeo['sameState'] = deliveredGeo['seller_state'] == deliveredGeo['customer_state']

print(deliveredGeo.groupby('sameState')['delivery_days'].mean())
print(deliveredGeo.groupby('sameState')['delivery_delay_days'].mean())
# STATUS: not yet run/confirmed.


sameState
False    14.586863
True      7.462180
Name: delivery_days, dtype: float64
sameState
False   -12.910477
True    -10.459267
Name: delivery_delay_days, dtype: float64


#### Q10. Which seller-state -> customer-state pairs show the worst average delay?

In [17]:
pairDelay = deliveredGeo.groupby(['seller_state', 'customer_state'])['delivery_delay_days'].mean()
pairCount = deliveredGeo.groupby(['seller_state', 'customer_state']).size()
pairSummary = pd.DataFrame({'avgDelay': pairDelay, 'orderCount': pairCount})

# Drop tiny pairs before ranking - a 2-order pair's "average" isn't reliable.
pairSummary = pairSummary[pairSummary['orderCount'] >= 30]
print(pairSummary.sort_values('avgDelay', ascending=False).head(15))
# STATUS: not yet run/confirmed.


                              avgDelay  orderCount
seller_state customer_state                       
PR           AL              -4.325581          43
RJ           CE              -5.125000          56
DF           DF              -7.573770          61
SP           AL              -8.241758         273
GO           GO              -9.179487          39
MG           SE              -9.365854          41
SP           SE              -9.375527         237
BA           BA              -9.418919          74
SP           MA              -9.551913         549
MA           SP              -9.561538         130
SP           ES              -9.600247        1621
RJ           MS              -9.651163          43
SP           SP             -10.220901       35425
RS           BA             -10.265306          49
PR           BA             -10.300613         163


#### Q11. Do seller-sparse customer states see longer delivery times?

In [18]:
# TO, AL, RR, AP - no local seller presence, per the raw audit.
sparseStates = ['TO', 'AL', 'RR', 'AP']
print(deliveredGeo[deliveredGeo['customer_state'].isin(sparseStates)]['delivery_days'].mean())
print(deliveredGeo[~deliveredGeo['customer_state'].isin(sparseStates)]['delivery_days'].mean())
# STATUS: not yet run/confirmed.


22.041666666666668
11.930825897963853


## Section D— Delivery Trends Over Time

#### Q12. Has average total delivery time trended up, down, or flat?

In [19]:
monthlyDeliveryDaysFull = deliveredOrders.groupby('purchaseMonth')['delivery_days'].mean()
print(monthlyDeliveryDaysFull)
# STATUS: not yet run/confirmed - note the same 2016-09/12 and 2018-09/10
# exclusions from the Section B population check apply here too.


purchaseMonth
2016-09    54.000000
2016-10    19.214559
2016-12     4.000000
2017-01    12.084112
2017-02    12.615990
2017-03    12.398428
2017-04    14.352584
2017-05    10.762698
2017-06    11.506382
2017-07    11.152292
2017-08    10.704605
2017-09    11.400723
2017-10    11.416927
2017-11    14.699506
2017-12    14.936695
2018-01    13.637290
2018-02    16.508619
2018-03    15.867485
2018-04    11.048544
2018-05    10.959105
2018-06     8.774442
2018-07     8.503736
2018-08     7.286412
Freq: M, Name: delivery_days, dtype: float64


#### Q13. Is there a seasonal pattern by calendar month, regardless of year?

In [20]:
seasonalDeliveryDays = deliveredOrders.groupby(deliveredOrders['order_purchase_timestamp'].dt.month)['delivery_days'].mean()
print(seasonalDeliveryDays)
# STATUS: not yet run/confirmed. Caveat to check: the dataset only spans
# ~2 full years, so a "seasonal" read here has very little repeat-year
# evidence behind it - treat as exploratory, not a confirmed pattern.


order_purchase_timestamp
1     13.488488
2     15.725445
3     14.942815
4     11.884628
5     10.891480
6      9.702059
7      9.524608
8      8.645323
9     11.410985
10    11.846381
11    14.699506
12    14.934712
Name: delivery_days, dtype: float64


In [22]:
sameStateSummary = deliveredGeo.groupby('sameState').agg(
    meanDeliveryDays=('delivery_days', 'mean'),
    medianDeliveryDays=('delivery_days', 'median')
)
sameStateSummary.to_csv('../data/summary/same_state_delivery.csv')

monthlyLateRate.to_csv('../data/summary/monthly_late_rate.csv', header=True)